# 9.8 Python

Chapter 9 develops conditional expectation, introduces the idea of prediction via linear regression, and works through several striking examples involving expected waiting times. This section uses Python to simulate the mystery prize problem, compare the waiting times to see `HH` versus `HT` in a sequence of coin tosses, and fit a simple linear regression by formula.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Mystery Prize Simulation

In Example 9.1.7, a mystery prize has unknown value $V \sim \text{Unif}(0, 1)$. You submit a bid $b$; the bid is accepted if $b > \frac{2}{3}V$, i.e., if $V < \frac{3}{2}b$. When the bid is accepted you receive the prize and pay $b$, so your profit is $V - b$. The surprising result is that the expected profit conditional on winning is always negative, regardless of the bid.

To see why, note that conditioning on acceptance means $V < \frac{3}{2}b$, so $V \mid \text{accepted} \sim \text{Unif}(0, \frac{3}{2}b)$, with conditional expectation $\frac{3}{4}b$. The expected profit is $\frac{3}{4}b - b = -\frac{1}{4}b < 0$.

Simulation confirms this for any bid. We generate $10^5$ prize values and keep only those where the bid was accepted, then compute the average profit.

In [ ]:
nsim = 10**5
v = rng.uniform(size=nsim)

b = 0.6
accepted = b > (2/3) * v
avg_profit = np.mean(v[accepted]) - b

print(f"Bid b = {b}")
print(f"Acceptance rate:          {accepted.mean():.4f}  (true: 3b/2 = {min(1.5*b, 1):.4f})")
print(f"E[V | accepted]:          {np.mean(v[accepted]):.4f}  (true: 3b/4 = {0.75*b:.4f})")
print(f"Expected profit | won:    {avg_profit:.4f}  (true: -b/4 = {-b/4:.4f})")

The expected profit is close to $-b/4 = -0.15$, confirming the theory. This holds for every bid — we can check a range of values.

In [ ]:
bids = np.arange(0.1, 1.0, 0.1)
v = rng.uniform(size=nsim)

print(f"{'Bid':>6}  {'Avg profit | won':>18}  {'True -b/4':>12}")
print("-" * 42)
for b in bids:
    mask = b > (2/3) * v
    profit = np.mean(v[mask]) - b if mask.any() else float("nan")
    print(f"{b:6.1f}  {profit:18.4f}  {-b/4:12.4f}")

Every bid yields a negative expected profit. Higher bids win more often but the winner's curse grows proportionally: you pay more and the conditional prize value is only $3/4$ of your bid.

## Time Until HH vs. HT

Example 9.1.9 establishes a striking asymmetry: the expected number of fair coin flips until `HH` first appears is **6**, while the expected waiting time until `HT` first appears is only **4**. Both patterns require seeing two specific consecutive flips, yet the expected waiting times differ.

To simulate this, we generate long strings of `H`s and `T`s and use Python's `str.find` to locate the first occurrence of each pattern. If the pattern starts at 0-indexed position $i$, it ends at position $i + 1$, so the waiting time (number of flips until the pattern is complete) is `find(pattern) + len(pattern)`.

In [ ]:
n_seq = 10**3
seq_len = 200   # long enough that HH and HT are virtually certain to appear

sequences = [
    "".join(rng.choice(["H", "T"], size=seq_len))
    for _ in range(n_seq)
]

wt_hh = [s.find("HH") + 2 for s in sequences]
wt_ht = [s.find("HT") + 2 for s in sequences]

print(f"Mean waiting time for HH: {np.mean(wt_hh):.2f}  (true: 6)")
print(f"Mean waiting time for HT: {np.mean(wt_ht):.2f}  (true: 4)")

The simulated means are close to 6 and 4. To see the full distribution of both waiting times, we can plot side-by-side histograms.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

sns.histplot(wt_hh, binwidth=1, ax=axes[0])
axes[0].axvline(np.mean(wt_hh), color="black", linestyle="--", label=f"mean = {np.mean(wt_hh):.1f}")
axes[0].set_xlabel("Flips until HH")
axes[0].set_ylabel("Count")
axes[0].set_title("Waiting time for HH")
axes[0].legend()

sns.histplot(wt_ht, binwidth=1, ax=axes[1])
axes[1].axvline(np.mean(wt_ht), color="black", linestyle="--", label=f"mean = {np.mean(wt_ht):.1f}")
axes[1].set_xlabel("Flips until HT")
axes[1].set_title("Waiting time for HT")
axes[1].legend()

plt.tight_layout()
plt.show()

The distributions have different shapes, not just different means. The `HT` waiting time has a lighter right tail because once an `H` appears, the pattern completes the moment the next flip is a `T`. The `HH` waiting time has a heavier tail because a `T` after a run of `H`s resets the progress entirely.

## Linear Regression

Example 9.3.10 derives the slope and intercept of the least-squares regression line for predicting $Y$ from $X$:

$$\hat{b} = \frac{\mathrm{Cov}(X, Y)}{\mathrm{Var}(X)}, \qquad \hat{a} = E(Y) - \hat{b}\, E(X).$$

In practice we substitute sample quantities for the population ones. To check that these formulas recover the true parameters, we simulate data from a known model: $Y = 3 + 5X + \varepsilon$ where $X \sim N(0,1)$ and $\varepsilon \sim N(0,1)$ independently, so the true intercept is $a = 3$ and the true slope is $b = 5$.

In [ ]:
n = 100
x = rng.normal(size=n)
y = 3 + 5 * x + rng.normal(size=n)

# np.cov returns the 2x2 sample covariance matrix; [0,1] is Cov(x, y)
cov_matrix = np.cov(x, y)   # uses ddof=1 by default
b_hat = cov_matrix[0, 1] / cov_matrix[0, 0]
a_hat = np.mean(y) - b_hat * np.mean(x)

print(f"Estimated slope:     {b_hat:.4f}  (true: 5)")
print(f"Estimated intercept: {a_hat:.4f}  (true: 3)")

Both estimates are close to their true values. Plotting the data with the fitted line makes the quality of the fit visible.

In [ ]:
x_line = np.array([x.min(), x.max()])
y_line = a_hat + b_hat * x_line

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(x=x, y=y, ax=ax, alpha=0.6, s=25, label="Data")
sns.lineplot(x=x_line, y=y_line, ax=ax,
             label=rf"$\hat{{y}} = {a_hat:.2f} + {b_hat:.2f}x$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Simulated data with least-squares regression line")
ax.legend()
plt.tight_layout()
plt.show()

The regression line passes through the center of the point cloud and captures the strong positive linear trend. The scatter around the line comes from the $\varepsilon \sim N(0,1)$ noise term; with $b = 5$ the signal-to-noise ratio is high, so 100 observations are more than enough to estimate both parameters accurately.